In [1]:
# ============================================================
# CELL 1: Install Required Dependencies
# ============================================================
!pip install nbformat nbconvert ipykernel pandas openpyxl -q

In [2]:
# Discover available kernel names
import subprocess, json
result = subprocess.run(["jupyter", "kernelspec", "list", "--json"], capture_output=True, text=True)
kernels = json.loads(result.stdout)
for name, info in kernels["kernelspecs"].items():
    print(f"{name:40} -> {info['spec']['display_name']}")

python3                                  -> Python 3 (ipykernel)


In [3]:
# ============================================================
# CELL 2 (FIXED v3): Run All Scraper Notebooks via nbconvert
# Injects a bootstrap cell to fix 'Path not defined' errors
# ============================================================
import os, glob, time
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor, CellExecutionError
from pathlib import Path
from datetime import datetime

SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
RUN_LOG = []

# Bootstrap cell to inject at start of every notebook
# Ensures all common imports are available from cell 1 onwards
BOOTSTRAP_CODE = '''# --- AUTO-INJECTED BOOTSTRAP (by orchestrator) ---
import os, re, time, json
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
print("[Bootstrap] Common imports loaded")
'''

# All 17 scraper notebooks
NOTEBOOKS = sorted([
    nb for nb in SCRIPTS_DIR.glob("*.ipynb")
    if "MASTER" not in nb.name
])

print(f"Found {len(NOTEBOOKS)} scraper notebooks:")
for nb in NOTEBOOKS:
    print(f"  - {nb.name}")

print("\n" + "="*60)
print(f"Starting execution at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60 + "\n")

# FORCE python3 kernel - avoids 'No such kernel' errors
KERNEL = "python3"

for nb_path in NOTEBOOKS:
    start = time.time()
    print(f"[RUNNING] {nb_path.name} ...", end=" ", flush=True)
    try:
        with open(nb_path, encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)
        
        # Inject bootstrap cell at position 0
        bootstrap_cell = nbformat.v4.new_code_cell(BOOTSTRAP_CODE)
        nb.cells.insert(0, bootstrap_cell)
        
        ep = ExecutePreprocessor(
            timeout=600,
            kernel_name=KERNEL,
            allow_errors=True  # allow cell-level errors, don't stop the notebook
        )
        ep.preprocess(nb, {"metadata": {"path": str(SCRIPTS_DIR)}})
        
        # Remove the injected bootstrap cell before saving
        nb.cells.pop(0)
        
        # Save the executed notebook back (preserves outputs)
        with open(nb_path, "w", encoding="utf-8") as f:
            nbformat.write(nb, f)
        
        elapsed = round(time.time() - start, 1)
        print(f"DONE ({elapsed}s)")
        RUN_LOG.append({"notebook": nb_path.name, "status": "SUCCESS", "elapsed_s": elapsed})
    except CellExecutionError as e:
        elapsed = round(time.time() - start, 1)
        print(f"FAILED ({elapsed}s) - {str(e)[:120]}")
        RUN_LOG.append({"notebook": nb_path.name, "status": "FAILED", "elapsed_s": elapsed})
    except Exception as e:
        elapsed = round(time.time() - start, 1)
        print(f"ERROR ({elapsed}s) - {str(e)[:120]}")
        RUN_LOG.append({"notebook": nb_path.name, "status": "ERROR", "elapsed_s": elapsed})

print("\n" + "="*60)
print("EXECUTION SUMMARY")
print("="*60)
for log in RUN_LOG:
    icon = "✅" if log["status"] == "SUCCESS" else "❌"
    print(f"{icon} {log['notebook']:<55} {log['status']:<10} {log['elapsed_s']}s")

success_count = sum(1 for l in RUN_LOG if l["status"] == "SUCCESS")
print(f"\nTotal: {success_count}/{len(RUN_LOG)} notebooks completed successfully")

Found 30 scraper notebooks:
  - Accenture India Job Scraper.ipynb
  - American Express India Job Scraper.ipynb
  - Apple india Job scrapper.ipynb
  - Atlassian India Job Scraper.ipynb
  - Capgemini India Job Scraper.ipynb
  - Cognizant India Job Scraper.ipynb
  - Continental India Job Scraper.ipynb
  - Eli Lilly India Job Scraper.ipynb
  - Fidelity Investments India Job Scraper.ipynb
  - Goldman Sachs India Job Scraper.ipynb
  - Google India Job Scrapper.ipynb
  - HCL Technologies India Job Scraper.ipynb
  - IBM India India Job Scraper.ipynb
  - Infosys India Job Scraper.ipynb
  - Loreal India Job Scraper.ipynb
  - MSCI India Job Scraper.ipynb
  - Mastercard India Job Scraper.ipynb
  - Microsoft India Job Scrapper.ipynb
  - Morgan Stanley India Job Scraper.ipynb
  - Novartis India Job scrapper.ipynb
  - RTX India Job Scraper.ipynb
  - Salesforce India Job Scraper.ipynb
  - Sanofi India Job Scrapper.ipynb
  - ServiceNow India Job Scraper.ipynb
  - Stripe India Job Scraper.ipynb
  - Syng

/opt/anaconda3/envs/JobAnalyser/lib/python3.14/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


ERROR (602.3s) - A cell timed out while it was being executed, after 600 seconds.
The message was: Cell execution timed out.
Here is a pr
[RUNNING] American Express India Job Scraper.ipynb ... DONE (6.0s)
[RUNNING] Apple india Job scrapper.ipynb ... DONE (273.1s)
[RUNNING] Atlassian India Job Scraper.ipynb ... DONE (39.3s)
[RUNNING] Capgemini India Job Scraper.ipynb ... DONE (49.0s)
[RUNNING] Cognizant India Job Scraper.ipynb ... DONE (7.5s)
[RUNNING] Continental India Job Scraper.ipynb ... DONE (91.0s)
[RUNNING] Eli Lilly India Job Scraper.ipynb ... DONE (33.7s)
[RUNNING] Fidelity Investments India Job Scraper.ipynb ... DONE (41.7s)
[RUNNING] Goldman Sachs India Job Scraper.ipynb ... DONE (33.3s)
[RUNNING] Google India Job Scrapper.ipynb ... DONE (29.7s)
[RUNNING] HCL Technologies India Job Scraper.ipynb ... DONE (17.6s)
[RUNNING] IBM India India Job Scraper.ipynb ... DONE (48.7s)
[RUNNING] Infosys India Job Scraper.ipynb ... DONE (56.6s)
[RUNNING] Loreal India Job Scraper.ipynb ... D

In [4]:
import pandas as pd
import datetime
from pathlib import Path

# ----- Canonical 24-column schema (matches scraper_utils.py) -----
COLS = [
    "job_id", "title", "company_name", "raw_jd_text",
    "skills_required", "skills_preferred",
    "min_years_experience", "max_years_experience",
    "seniority_level", "location_city", "location_country",
    "work_mode", "employment_type",
    "degree_required", "degree_preferred_field",
    "industry", "salary_min", "salary_max", "salary_currency",
    "date_posted", "is_active",
    "job_url", "business_unit", "source_platform"
]

# ----- Valid controlled-vocab values -----
VALID_SENIORITY  = {"junior", "mid", "senior", "lead"}
VALID_WORK_MODE  = {"onsite", "hybrid", "remote"}
VALID_EMP_TYPE   = {"full-time", "part-time", "contract", "internship"}

# ----- Search all company output folders for CSVs -----
BASE_DIR   = Path.home() / "Job_Scrapers"
MONTH_TAG  = datetime.datetime.now().strftime("%Y_%m")

csv_files  = list(BASE_DIR.rglob(f"*/{MONTH_TAG}/*.csv"))
# Fallback: pick any CSV in Outputs/ subdirs
if not csv_files:
    csv_files = list(BASE_DIR.rglob("*/Outputs/**/*.csv"))
if not csv_files:
    csv_files = list(BASE_DIR.rglob("*.csv"))
    csv_files = [f for f in csv_files if "MASTER" not in f.name]

print(f"Found {len(csv_files)} CSV file(s) to merge:")
for f in csv_files:
    print(f"  {f}")

# ----- Load & tag source file -----
frames = []
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path, dtype=str)
        df["_source_file"] = csv_path.name
        frames.append(df)
        print(f"  Loaded {len(df):>5} rows  <- {csv_path.name}")
    except Exception as e:
        print(f"  SKIP {csv_path.name}: {e}")

if not frames:
    print("\nNO CSV DATA FOUND. Run the scrapers first (Cell 2).")
else:
    # ----- Merge -----
    raw = pd.concat(frames, ignore_index=True)

    # ----- Ensure all canonical columns exist -----
    for col in COLS:
        if col not in raw.columns:
            raw[col] = pd.NA

    # ----- Clean job_url: must start with http -----
    if "job_url" in raw.columns:
        raw["job_url"] = raw["job_url"].astype(str).str.strip()
        raw.loc[~raw["job_url"].str.startswith("http", na=False), "job_url"] = pd.NA
        print(f"  job_url populated: {raw['job_url'].notna().sum()} / {len(raw)} rows")

    # ----- Controlled vocab normalisation -----
    raw["seniority_level"]  = raw["seniority_level"].str.lower().where(
        raw["seniority_level"].str.lower().isin(VALID_SENIORITY))
    raw["work_mode"]        = raw["work_mode"].str.lower().where(
        raw["work_mode"].str.lower().isin(VALID_WORK_MODE))
    raw["employment_type"]  = raw["employment_type"].str.lower().where(
        raw["employment_type"].str.lower().isin(VALID_EMP_TYPE))

    # ----- Select & reorder to canonical schema -----
    out = raw[COLS].copy()

    # ----- Save -----
    out_dir = BASE_DIR / "Master_Output"
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_out  = out_dir / f"ALL_JOBS_NORMALIZED_{MONTH_TAG}.csv"
    xlsx_out = out_dir / f"ALL_JOBS_NORMALIZED_{MONTH_TAG}.xlsx"

    out.to_csv(csv_out,   index=False)
    out.to_excel(xlsx_out, index=False)

    print(f"\nDone! {len(out)} total jobs saved.")
    print(f"  CSV  -> {csv_out}")
    print(f"  XLSX -> {xlsx_out}")
    print(f"  Columns ({len(out.columns)}): {list(out.columns)}")
    print(out[["company_name","title","job_url"]].head(10).to_string(index=False))

Found 75 CSV file(s) to merge:
  /Users/incognito/Job_Scrapers/Apple/Outputs/2026_03/Apple_jobs_2026-03-24.csv
  /Users/incognito/Job_Scrapers/Apple/Outputs/2026_03/Apple_jobs_FULL_2026-03-24.csv
  /Users/incognito/Job_Scrapers/Apple/Outputs/2026_03/Apple_jobs_2026-03-22.csv
  /Users/incognito/Job_Scrapers/Apple/Outputs/2026_03/Apple_jobs_2026-03-21.csv
  /Users/incognito/Job_Scrapers/Apple/Outputs/2026_03/Apple_jobs_FULL_2026-03-22.csv
  /Users/incognito/Job_Scrapers/Capgemini/Outputs/2026_03/Capgemini_jobs_FULL_2026-03-22.csv
  /Users/incognito/Job_Scrapers/Capgemini/Outputs/2026_03/Capgemini_jobs_2026-03-22.csv
  /Users/incognito/Job_Scrapers/Synopsys/Outputs/2026_03/Synopsys_jobs_FULL_2026-03-24.csv
  /Users/incognito/Job_Scrapers/Synopsys/Outputs/2026_03/Synopsys_jobs_2026-03-24.csv
  /Users/incognito/Job_Scrapers/TCS/Outputs/2026_03/TCS_jobs_FULL_2026-03-22.csv
  /Users/incognito/Job_Scrapers/TCS/Outputs/2026_03/TCS_jobs_FULL_2026-03-24.csv
  /Users/incognito/Job_Scrapers/TCS/Out

In [5]:
# ============================================================
# CELL 4: Schema Validation & Quality Report
# ============================================================
import pandas as pd

try:
    master
except NameError:
    print("Run Cell 3 first to generate the master DataFrame.")
else:
    print("\n" + "="*60)
    print("SCHEMA VALIDATION REPORT")
    print("="*60)

    # Column completeness
    print(f"\n{'COLUMN':<30} {'NON-NULL':>10} {'NULL%':>8} {'DTYPE':<15}")
    print("-"*65)
    for col in COLS:
        non_null = master[col].notna().sum()
        null_pct  = round((1 - non_null / len(master)) * 100, 1)
        dtype     = str(master[col].dtype)
        flag = " <-- MISSING" if null_pct > 80 else ""
        print(f"{col:<30} {non_null:>10} {null_pct:>7}% {dtype:<15}{flag}")

    print("\n" + "="*60)
    print("CONTROLLED VOCAB DISTRIBUTION")
    print("="*60)
    for col in ["seniority_level", "work_mode", "employment_type"]:
        print(f"\n{col}:")
        print(master[col].value_counts(dropna=False).to_string())

    print("\n" + "="*60)
    print("JOBS PER COMPANY")
    print("="*60)
    print(master["company_name"].value_counts().to_string())

    print("\n" + "="*60)
    print("ACTIVE vs INACTIVE")
    print("="*60)
    print(master["is_active"].value_counts().to_string())

    print("\n" + "="*60)
    print(f"TOTAL: {len(master)} unique jobs across {master['company_name'].nunique()} companies")
    print(f"Output: ~/Job_Scrapers/Master_Output/")
    print("="*60)

Run Cell 3 first to generate the master DataFrame.
